In [7]:
import torch
import numpy as np
from src.data.sequence import Sequence  # 替换为你的实际模块路径

# 构造示例数据
t_start = 0.0
arrival_times = np.array([1.0, 2.5, 4.0, 6.0])  # 事件发生时间
t_end = 8.0

# 计算 inter_times: Δt_0, Δt_1, ..., Δt_N, Δt_survival
inter_times = np.diff(arrival_times, prepend=[t_start], append=[t_end])  # shape = [5]

# 额外属性，如震级 (mag) 和位置 (loc)
mag = np.array([3.1, 2.8, 4.0, 3.5])  # shape = [4]
loc = np.array([
    [30.5, 102.3],
    [30.6, 102.4],
    [30.7, 102.5],
    [30.8, 102.6],
])  # shape = [4, 2]

# 创建 Sequence 对象
seq = Sequence(
    inter_times=inter_times,
    t_start=t_start,
    mag=mag,
    loc=loc
)

# ✅ 查看完整事件序列
print("Original arrival times:", seq.arrival_times)
print("Original inter_times:", seq.inter_times)
print("Original magnitude:", seq['mag'])

# ✅ 调用 get_subsequence（例如截取 [2.0, 5.0] 之间的事件）
sub_seq = seq.get_subsequence(start=2.0, end=5.0)

# ✅ 输出子序列内容
print("\n--- Subsequence ---")
print("Arrival times:", sub_seq.arrival_times)
print("Inter times:", sub_seq.inter_times)
print("Magnitude:", sub_seq['mag'])
print("Location:", sub_seq['loc'])


Original arrival times: tensor([1.0000, 2.5000, 4.0000, 6.0000], dtype=torch.float64)
Original inter_times: tensor([1.0000, 1.5000, 1.5000, 2.0000, 2.0000], dtype=torch.float64)
Original magnitude: tensor([3.1000, 2.8000, 4.0000, 3.5000], dtype=torch.float64)

--- Subsequence ---
Arrival times: tensor([2.5000, 4.0000], dtype=torch.float64)
Inter times: tensor([0.5000, 1.5000, 1.0000], dtype=torch.float64)
Magnitude: tensor([2.8000, 4.0000], dtype=torch.float64)
Location: tensor([[ 30.6000, 102.4000],
        [ 30.7000, 102.5000]], dtype=torch.float64)


In [8]:
sub_event_seq = sub_seq.to_event_sequence()

In [9]:
print(sub_event_seq.arrival_times,sub_event_seq.inter_times)

tensor([0.0000, 1.5000]) tensor([0.0000, 1.5000])


In [10]:
from src.data.tpp_dataset import TppDataset
ds = TppDataset([seq])

In [11]:
dl = ds.get_dataloader(batch_size=2, shuffle=True)

In [12]:
for batch in dl:
    print("\n--- Batch ---")
    print("Arrival times:", batch['arrival_times'])
    print("Inter times:", batch['inter_times'])
    print("Magnitude:", batch['mag'])
    print("Location:", batch['loc'])
    break  # 只打印第一个批次


--- Batch ---
Arrival times: tensor([[1.0000, 2.5000, 4.0000, 6.0000, 8.0000]], dtype=torch.float64)
Inter times: tensor([[1.0000, 1.5000, 1.5000, 2.0000, 2.0000]], dtype=torch.float64)
Magnitude: tensor([[3.1000, 2.8000, 4.0000, 3.5000, 0.0000]], dtype=torch.float64)
Location: tensor([[[ 30.5000, 102.3000],
         [ 30.6000, 102.4000],
         [ 30.7000, 102.5000],
         [ 30.8000, 102.6000],
         [  0.0000,   0.0000]]], dtype=torch.float64)
